In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
from torch import Tensor
from torch.nn import Linear, Sequential, Sigmoid
from torch.nn.functional import binary_cross_entropy, binary_cross_entropy_with_logits, sigmoid
from torch.optim import SGD


import import_ipynb
from common import assert_eq, assert_ge, check_eq, Patient, T # type: ignore


# Convert Patient from 3-classes {healthy, viral, bacterial} to 2-classes {healthy, sick}
def Patient2C(healthy: float) -> Patient:
    #
    # `bacterial` is preferred to demonstrate CRP scaling
    #

    if bacterial := 1:
        patient = Patient([healthy, 0.0, 1 - healthy])
        patient.data[-1] = int(0.5 * patient.data[-1]) # 0→0, 2→1
        return patient
    else:
        return Patient([healthy, 1 - healthy, 0])


def logistic_regression_2c_sgd_autograd(X: Tensor, Y: Tensor, epochs=2000, lr=0.1) -> tuple[float, callable]:
    """
    Perform logistic regression using Stochastic Gradient Descent (SGD) with autograd.

    Args:
        X: Input features of shape (Samples, Features)
        y: Target values of shape (Samples, 1)
        epochs: Number of training epochs
        lr: Learning rate

    Returns:
        A tuple containing the final loss and a prediction function that takes new input data and returns predicted probabilities.
    """

    (s, f) = X.shape

    # `BCEWithLogitsLoss` combines a `Sigmoid` layer and the `BCELoss` in one single class. 
    # This version is more numerically stable than using a plain `Sigmoid` followed by 
    # a `BCELoss` as, by combining the operations into one layer, we take advantage of 
    # the log-sum-exp trick for numerical stability.
    model = Linear(f, 1)
    loss_fn = binary_cross_entropy_with_logits

    # model = Sequential(Linear(f, 1), Sigmoid())
    # loss_fn = binary_cross_entropy

    optimizer = SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        optimizer.zero_grad()

        Z = model(X)
        assert_eq(Z.shape, (s, 1))

        L = loss_fn(Z, Y)
        assert_eq(L.shape, ())
        
        L.backward()
        optimizer.step()

    if isinstance(model, Linear):
        return (L.item(), lambda x: model(x).sigmoid())
    else:
        return (L.item(), model)


def _test_logistic_regression_2c_sgd_autograd(epochs=2000, lr=0.1) -> None:
    """
    Test the logistic regression implementation on a synthetic dataset of patients. 
    The dataset consists of two classes: healthy (0) and sick (1). 
    The model is trained on 80 samples and then evaluated on 20 samples (10 healthy and 10 sick). 
    The test checks if the model achieves at least 90% accuracy on the evaluation set. 
    """
    
    training_data = T([Patient2C(0.5).data for _ in range(80)])

    X = training_data[:, :-1]
    X[:, 0] /= 100 # Temperature scaling
    X[:, 5] /= 200 # CRP scaling    

    Y = training_data[:, -1].unsqueeze(1)

    (_, model) = logistic_regression_2c_sgd_autograd(X, Y, epochs, lr)

    total = 0
    correct = 0

    for d in ([Patient2C(0.0).data for _ in range(10)] + 
              [Patient2C(1.0).data for _ in range(10)]):
        X = T(d[:-1])
        Y = T(d[-1])

        X[0] /= 100
        X[5] /= 200

        total += 1
        correct += check_eq((model(X) >= 0.5), [Y])
        
    assert_ge(correct / total, 0.9)


if __name__ == "__main__":
    _test_logistic_regression_2c_sgd_autograd()